# Lab01 - sprint 1
<a id="top"></a>

**Датасет:** рекламные клики пользователей (1000 строк, 10 столбцов): время на сайте, возраст, доход района, интернет-активность, пол, страна, время визита и целевой признак `Clicked on Ad` (кликнул на рекламу или нет).

**Три случайные величины:**
1. `Daily Time Spent on Site` - время на сайте за день;
2. `Daily Internet Usage` - интернет-активность за день (по смыслу того же типа, что и первая: время);
3. `Age` - возраст пользователя.

Единицы измерения в датасете не указаны. Считаем, что время в первых двух столбцах измеряется в **минутах** - это наше допущение, оно не проверяемо.

Данные берутся из файла `logistic_Regression.csv`, лежащего рядом с ноутбуком.


## Оглавление
- [Датасет и подготовка](#data)
- [Random Variable 01 - время на сайте](#rv1): [описание](#rv1-desc) · [EDA и распределение](#rv1-eda) · [независимость и оценки](#rv1-ind) · [гипотеза](#rv1-hyp)
- [Random Variable 02 - интернет-активность](#rv2): [описание](#rv2-desc) · [EDA и распределение](#rv2-eda) · [независимость и оценки](#rv2-ind)
- [Random Variable 03 - возраст](#rv3): [описание](#rv3-desc) · [EDA и распределение](#rv3-eda) · [независимость и оценки](#rv3-ind)
- [Lab resume (sprint 1)](#s1-resume)
- [Lab01 - sprint 2](#sprint2)


<a id="data"></a>
## Датасет и подготовка
Загружаем данные и сразу проверяем, что с ними всё в порядке.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from IPython.display import display, Markdown

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})
C1, C2 = '#3b6ea5', '#d1603d'   # единые цвета для всех графиков

df = pd.read_csv('logistic_Regression.csv')
T, I, A = 'Daily Time Spent on Site', 'Daily Internet Usage', 'Age'
LABELS = {T: 'Время на сайте, мин', I: 'Интернет за день, мин', A: 'Возраст, лет'}

print('Размер таблицы:', df.shape)
print('Пропусков:', int(df.isna().sum().sum()), '| дубликатов строк:', int(df.duplicated().sum()))
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'logistic_Regression.csv'

In [ ]:
df.dtypes.to_frame('тип')

In [ ]:
# признаки того, что данные синтетические (проверяем сами)
checks = pd.Series({
    'Кликнули / не кликнули': f"{(df['Clicked on Ad']==1).sum()} / {(df['Clicked on Ad']==0).sum()}",
    'Среднее времени на сайте': round(df[T].mean(), 4),
    'Среднее интернет-времени': round(df[I].mean(), 4),
    'Средний доход района': round(df['Area Income'].mean(), 2),
    'Уникальных Ad Topic Line': df['Ad Topic Line'].nunique(),
    'Уникальных городов': df['City'].nunique(),
})
checks.to_frame('значение')

**Что видно по этим цифрам.** Пропусков и дубликатов нет. Зато есть признаки того, что данные **не настоящие, а сгенерированные**: клики разделены ровно 500 на 500, средние времени на сайте, интернет-времени и дохода равны круглым числам (65.00, 180.00, 55000.00), все 1000 строк `Ad Topic Line` уникальны, а города похожи на случайные имена. Это важно для выводов: всё, что мы узнаем про связи между признаками, относится к этому набору данных, а не к реальной аудитории. Мы будем возвращаться к этому в выводах.

Ниже - функции, которые используются для всех трёх величин (чтобы не копировать один и тот же код три раза).

In [ ]:
# вспомогательные функции для всех трёх величин

def randomness_check(col, plot=True):
    """Можно ли получить значение формулой? Линейная регрессия на остальные признаки."""
    others = [c for c in [T, I, A, 'Area Income', 'Male'] if c != col]
    y = df[col].to_numpy(float)

    def fit(cols):
        X = np.column_stack([np.ones(len(df))] + [df[c] for c in cols])
        b = np.linalg.lstsq(X, y, rcond=None)[0]
        res = y - X @ b
        return 1 - res.var() / y.var(), res, X @ b

    r2, res, pred = fit(others)
    r2_click, _, _ = fit(others + ['Clicked on Ad'])

    lag1 = np.corrcoef(y[:-1], y[1:])[0, 1]
    s = (y > np.median(y)).astype(int)                      # критерий серий
    runs = 1 + (s[1:] != s[:-1]).sum(); n1 = s.sum(); n2 = len(s) - n1
    mu = 2 * n1 * n2 / (n1 + n2) + 1
    var = (mu - 1) * (mu - 2) / (n1 + n2 - 1)
    p_runs = 2 * (1 - stats.norm.cdf(abs((runs - mu) / np.sqrt(var))))

    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ax[0].scatter(pred, y, s=8, alpha=0.5, color=C1)
        ax[0].plot([y.min(), y.max()], [y.min(), y.max()], color=C2)
        ax[0].set(title='Факт против предсказания формулой', xlabel='Предсказание регрессии', ylabel=LABELS[col])
        sns.histplot(res, bins=25, kde=True, ax=ax[1], color=C1)
        ax[1].set(title='Остатки регрессии', xlabel='Остаток (то, что формула не объяснила)', ylabel='Число пользователей')
        plt.tight_layout(); plt.show()
    return pd.Series({'R2 (без Clicked on Ad)': r2, 'R2 (с Clicked on Ad)': r2_click,
                      'автокорреляция соседних строк': lag1, 'p критерия серий': p_runs}).round(4).to_frame(col)


def eda(col):
    x = df[col]
    s = pd.Series({'n': len(x), 'среднее': x.mean(), 'медиана': x.median(), 'станд. откл.': x.std(),
                   'дисперсия': x.var(), 'мин': x.min(), 'Q1': x.quantile(.25), 'Q3': x.quantile(.75),
                   'макс': x.max(), 'асимметрия': stats.skew(x), 'эксцесс': stats.kurtosis(x)})
    display(s.round(3).to_frame(col))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    sns.histplot(x, bins=25, kde=True, ax=ax[0], color=C1)
    ax[0].set(title='Гистограмма', xlabel=LABELS[col], ylabel='Число пользователей')
    sns.boxplot(x=x, ax=ax[1], color=C1)
    ax[1].set(title='Ящик с усами', xlabel=LABELS[col])
    stats.probplot(x, dist='norm', plot=ax[2])
    ax[2].set(title='QQ-график (против нормального)', xlabel='Теоретические квантили', ylabel='Выборочные квантили')
    plt.tight_layout(); plt.show()


def by_group(col):
    fig, ax = plt.subplots(figsize=(7, 4))
    for g, c, name in [(0, C1, 'не кликнули'), (1, C2, 'кликнули')]:
        sns.histplot(df[df['Clicked on Ad'] == g][col], bins=20, kde=True, color=c, alpha=0.45, label=name, ax=ax, stat='density')
    ax.set(title='Распределение по группам', xlabel=LABELS[col], ylabel='Плотность'); ax.legend()
    plt.tight_layout(); plt.show()


def normalization(col):
    x = df[col].to_numpy(float)
    z = (x - x.mean()) / x.std()
    mm = (x - x.min()) / (x.max() - x.min())
    tbl = pd.DataFrame({k: [v.mean(), v.std(), stats.skew(v), stats.kurtosis(v)] for k, v in
                        [('исходные', x), ('z-оценка', z), ('min-max', mm)]},
                       index=['среднее', 'станд. откл.', 'асимметрия', 'эксцесс'])
    display(tbl.round(4))
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    sns.histplot(x, bins=25, ax=ax[0], color=C1); ax[0].set(title='До', xlabel=LABELS[col], ylabel='Число пользователей')
    sns.histplot(z, bins=25, ax=ax[1], color=C2); ax[1].set(title='После z-нормализации', xlabel='Стандартные отклонения от среднего', ylabel='Число пользователей')
    plt.tight_layout(); plt.show()


def normality(col):
    rows = []
    for name, x in [('все', df[col]), ('не кликнули', df[df['Clicked on Ad'] == 0][col]), ('кликнули', df[df['Clicked on Ad'] == 1][col])]:
        rows.append({'группа': name, 'n': len(x), 'асимметрия': stats.skew(x), 'эксцесс': stats.kurtosis(x),
                     "p Д'Агостино": stats.normaltest(x).pvalue, 'p Шапиро-Уилка': stats.shapiro(x).pvalue})
    return pd.DataFrame(rows).set_index('группа').round(4)


def mixture_fit(col):
    """Смесь двух нормальных по группам Clicked on Ad."""
    x = df[col].to_numpy(float)
    g = df['Clicked on Ad'].to_numpy()
    w = [np.mean(g == k) for k in (0, 1)]
    m = [x[g == k].mean() for k in (0, 1)]
    s = [x[g == k].std(ddof=1) for k in (0, 1)]
    mix_pdf = lambda t: sum(wi * stats.norm.pdf(t, mi, si) for wi, mi, si in zip(w, m, s))
    mix_cdf = lambda t: sum(wi * stats.norm.cdf(t, mi, si) for wi, mi, si in zip(w, m, s))
    mean_th = sum(wi * mi for wi, mi in zip(w, m))
    var_th = sum(wi * (si**2 + mi**2) for wi, mi, si in zip(w, m, s)) - mean_th**2
    grid = np.linspace(x.min() - 5, x.max() + 5, 400)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(x, bins=30, stat='density', color=C1, alpha=0.4, ax=ax, label='данные')
    ax.plot(grid, stats.norm.pdf(grid, x.mean(), x.std(ddof=1)), color='gray', ls='--', label='одна нормальная')
    ax.plot(grid, mix_pdf(grid), color=C2, lw=2, label='смесь двух нормальных')
    ax.set(title='Подгонка распределения', xlabel=LABELS[col], ylabel='Плотность'); ax.legend()
    plt.tight_layout(); plt.show()
    res = pd.DataFrame({'значение': {
        'веса компонент': [round(v, 3) for v in w], 'средние компонент': [round(v, 2) for v in m],
        'станд. откл. компонент': [round(v, 2) for v in s],
        'среднее: смесь / данные': f"{mean_th:.2f} / {x.mean():.2f}",
        'дисперсия: смесь / данные': f"{var_th:.1f} / {x.var(ddof=0):.1f}",
        'KS p: одна нормальная': round(stats.kstest(x, stats.norm(x.mean(), x.std(ddof=1)).cdf).pvalue, 5),
        'KS p: смесь': round(stats.kstest(x, mix_cdf).pvalue, 5)}})
    return res


def consistency(col, reps=500, seed=0):
    """Состоятельность: оценки по выборкам растущего размера (с возвращением) против истинных параметров."""
    x = df[col].to_numpy(float)
    rng = np.random.default_rng(seed)
    mu, sig2, med = x.mean(), x.var(), np.median(x)
    rows = []
    for n in [10, 20, 50, 100, 200, 500, 1000, 2000, 5000]:
        s = x[rng.integers(0, len(x), (reps, n))]
        means, vs, meds = s.mean(1), s.var(1, ddof=1), np.median(s, 1)
        rows.append({'n': n, 'ср. ошибка среднего': np.abs(means - mu).mean(), 'разброс среднего': means.std(),
                     'теория sigma/sqrt(n)': np.sqrt(sig2 / n), 'разброс дисперсии': vs.std(), 'разброс медианы': meds.std(),
                     'смещение среднего': means.mean() - mu})
    tbl = pd.DataFrame(rows).set_index('n')
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].loglog(tbl.index, tbl['разброс среднего'], 'o-', color=C1, label='разброс среднего (опыт)')
    ax[0].loglog(tbl.index, tbl['теория sigma/sqrt(n)'], '--', color=C2, label='sigma / sqrt(n)')
    ax[0].loglog(tbl.index, tbl['разброс медианы'], 's-', color='gray', label='разброс медианы')
    ax[0].set(title='Разброс оценок убывает с ростом n', xlabel='Объём выборки n', ylabel='Стандартное отклонение оценки'); ax[0].legend()
    path = x[rng.integers(0, len(x), 5000)]
    ax[1].plot(np.cumsum(path) / np.arange(1, 5001), color=C1)
    ax[1].axhline(mu, color=C2, ls='--', label='истинное среднее')
    ax[1].set(title='Кумулятивное среднее одной серии', xlabel='Число наблюдений', ylabel=LABELS[col]); ax[1].legend()
    plt.tight_layout(); plt.show()
    return tbl.round(4)


def sampling_compare(col, n=100, reps=2000, seed=1):
    """Выборки по n из 1000 строк: с возвращением и без, reps повторений."""
    x = df[col].to_numpy(float); N = len(x)
    rng = np.random.default_rng(seed)
    rows = {}
    for name, rep in [('без возвращения', False), ('с возвращением', True)]:
        means, stds = [], []
        for _ in range(reps):
            s = rng.choice(x, n, replace=rep)
            means.append(s.mean()); stds.append(s.std(ddof=1))
        rows[name] = {'среднее выборочных средних': np.mean(means), 'разброс выборочных средних': np.std(means),
                      'средний выборочный std': np.mean(stds)}
    sig = x.std()
    rows['генеральная совокупность / теория'] = {'среднее выборочных средних': x.mean(),
        'разброс выборочных средних': f"{sig/np.sqrt(n):.4f} (с воз.), {sig/np.sqrt(n)*np.sqrt((N-n)/(N-1)):.4f} (без воз.)",
        'средний выборочный std': x.std(ddof=1)}
    return pd.DataFrame(rows).T


def pair_tests(ca, cb, plot=True):
    x, y = df[ca].to_numpy(float), df[cb].to_numpy(float)
    pr, sr = stats.pearsonr(x, y), stats.spearmanr(x, y)
    tab = pd.crosstab(pd.qcut(x, 5, labels=False, duplicates='drop'), pd.qcut(y, 5, labels=False, duplicates='drop'))
    chi2, p_chi, dof, _ = stats.chi2_contingency(tab)
    mi = mutual_info_regression(x.reshape(-1, 1), y, random_state=0)[0]
    rng = np.random.default_rng(0)
    mi0 = np.mean([mutual_info_regression(x.reshape(-1, 1), rng.permutation(y), random_state=0)[0] for _ in range(30)])
    if plot:
        fig, ax = plt.subplots(figsize=(6, 4.5))
        sns.regplot(x=x, y=y, scatter_kws={'s': 8, 'alpha': 0.4, 'color': C1}, line_kws={'color': C2}, ax=ax)
        ax.set(title=f'{LABELS[ca]} и {LABELS[cb]}', xlabel=LABELS[ca], ylabel=LABELS[cb])
        plt.tight_layout(); plt.show()
    return pd.Series({'Пирсон r': pr[0], 'p Пирсона': pr[1], 'Спирмен rho': sr[0], 'p Спирмена': sr[1],
                      'chi2 (сетка 5x5)': chi2, 'p chi2': p_chi, 'вз. информация': mi,
                      'вз. информация при перемешивании': mi0}).to_frame(f'{ca[:12]} / {cb[:12]}')


def click_test(col):
    a = df[df['Clicked on Ad'] == 1][col]; b = df[df['Clicked on Ad'] == 0][col]
    u = stats.mannwhitneyu(a, b, alternative='two-sided')
    w = stats.ttest_ind(a, b, equal_var=False)
    return pd.Series({'среднее (кликнули)': a.mean(), 'среднее (не кликнули)': b.mean(),
                      'медиана (кликнули)': a.median(), 'медиана (не кликнули)': b.median(),
                      'U Манна-Уитни': u.statistic, 'p Манна-Уитни': u.pvalue, 'p Уэлча': w.pvalue,
                      'P(у кликнувшего меньше)': 1 - u.statistic / (len(a) * len(b))}).to_frame(col)

<a id="rv1"></a>
## Random Variable 01 - время на сайте (`Daily Time Spent on Site`)

<a id="rv1-desc"></a>
### Description
**Природа события.** Это сколько времени (мин) пользователь проводит на сайте за день. Величина непрерывная, неотрицательная. Её значение зависит от множества причин: настроения, занятости, интереса к контенту, и заранее его по одному человеку не посчитать.

**Почему это случайная величина.** Случайная величина - это такая, значение которой нельзя получить детерминированной формулой по другим известным данным. Проверяем тремя способами:
1. Пробуем «выдать» значение формулой: линейная регрессия на все остальные числовые признаки (возраст, интернет-активность, доход района, пол). Если бы величина считалась формулой, коэффициент $R^2$ был бы близок к 1, а остатки были бы нулевыми.
2. Смотрим, нет ли закономерности по порядку строк: автокорреляция соседних значений и критерий серий. Если бы значения шли по какой-то схеме, соседние строки были бы связаны.
3. Смотрим на остатки: у настоящего «шума» они выглядят как случайный разброс вокруг нуля.

In [ ]:
randomness_check(T)

**Вывод.** Формула по остальным признакам объясняет всего $R^2 \approx 0.31$ разброса. Даже если добавить признак `Clicked on Ad`, получается 0.58, то есть почти половина разброса остаётся необъяснённой. Остатки широкие и не равны нулю. Соседние строки не связаны (автокорреляция 0.02, критерий серий $p = 0.95$, то есть гипотезу о случайном порядке не отвергаем). Значит, по этим данным значение времени на сайте детерминированной формулой не вычислить - это случайная величина.

**Оговорка:** у нас, скорее всего, синтетический датасет, так что «случайность» здесь - это свойство программы, которая данные создавала, а не живых людей.

<a id="rv1-eda"></a>
### EDA & Distribution
Описательные статистики, гистограмма с ядерной оценкой плотности, ящик с усами и QQ-график (если точки лежат на прямой, распределение нормальное).

In [ ]:
eda(T)

In [ ]:
by_group(T)

In [ ]:
normality(T)

**Выводы по EDA.**
- Среднее 65.0, медиана 68.2, стандартное отклонение 15.85, значения от 32.6 до 91.4 мин. Выбросов по ящику с усами нет.
- Асимметрия $-0.37$: хвост слева длиннее, большая часть пользователей сидит дольше среднего (медиана больше среднего).
- Эксцесс $-1.10$: распределение **более плоское**, чем нормальное (у нормального эксцесс 0). На гистограмме заметны два «горба», а QQ-график изгибается S-образно.
- Нормальность отвергается всеми тестами ($p \approx 0$).
- Причина плоской формы видна на графике по группам: у не кликнувших время около 76.9 мин ($\sigma = 7.6$), у кликнувших около 53.2 мин ($\sigma = 12.8$). Внутри групп эксцесс уже небольшой (+0.42 и −0.44), а общая картина - смесь двух кучек.

#### Нормализация
Делаем **z-нормализацию**: $z_i = \dfrac{x_i - \bar{x}}{s}$, где $\bar x$ - среднее, $s$ - стандартное отклонение. После неё среднее равно 0, а стандартное отклонение равно 1. Мотивация: величины с разными единицами и масштабом (минуты, годы, доллары) становятся сравнимыми, а многие методы (регрессия, расстояния, градиентный спуск) работают с ними лучше. Для сравнения показан также min-max: $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$, он переводит значения в отрезок [0, 1].

Важно: z-нормализация **не делает распределение нормальным**, она только сдвигает и растягивает ось. Поэтому асимметрия и эксцесс в таблице не меняются.

In [ ]:
normalization(T)

#### Математическая база распределения
**Нормальное распределение** $N(\mu, \sigma^2)$ - плотность
$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right),$$
математическое ожидание $\mu$, дисперсия $\sigma^2$, асимметрия 0, избыточный эксцесс 0. Колоколообразное, симметричное, около 95% значений лежит в пределах $\mu \pm 2\sigma$.

Наши данные не нормальные, поэтому берём **смесь двух нормальных распределений** (по группам `Clicked on Ad`):
$$f(x) = w_1\, f_1(x) + w_2\, f_2(x), \qquad f_k = N(\mu_k, \sigma_k^2), \quad w_1 + w_2 = 1.$$
Среднее и дисперсия смеси:
$$\mu = \sum_k w_k \mu_k, \qquad \sigma^2 = \underbrace{\sum_k w_k \sigma_k^2}_{\text{внутри групп}} + \underbrace{\sum_k w_k(\mu_k - \mu)^2}_{\text{между группами}}.$$
Если средние компонент $\mu_k$ сильно отличаются, плотность смеси получается плоской или двугорбой, а эксцесс отрицательным. Это как раз наш случай.

Для проверки согласия используем критерий Колмогорова-Смирнова с статистикой $D_n = \sup_x |F_n(x) - F(x)|$ ($F_n$ - эмпирическая функция распределения) [11]. Параметры мы оцениваем по тем же данным, поэтому $p$-value получается чуть завышенным (поправка Лиллиефорса [12]), это учитываем при выводах.

In [ ]:
mixture_fit(T)

**Вывод.** Одна нормальная кривая данные не описывает ($p \approx 0$ по KS). Смесь двух нормальных попадает в данные заметно лучше: среднее и дисперсия смеси совпадают с выборочными (65.00 и 251.3 против 251.1), $p = 0.067$ у KS, то есть на уровне 5% смесь не отвергается. У смеси есть недостаток: нормальные хвосты бесконечны, а реальные значения ограничены отрезком [32.6; 91.4]. Мы вернёмся к этому в sprint 2, когда будем строить датчик.

<a id="rv1-ind"></a>
### Independence & Estimations
#### Независимость
Время на сайте и интернет-активность - величины одной природы (обе время), поэтому проверяем их независимость. Для пары также смотрим время на сайте и возраст. Используем четыре способа:
- коэффициент Пирсона (линейная связь) и Спирмена (связь по рангам);
- $\chi^2$-тест независимости на таблице 5×5 (каждую величину разбили на 5 групп по квантилям; в каждой клетке ожидается около 40 человек);
- взаимная информация $I(X;Y)$: она равна 0 только для независимых величин; для сравнения считаем её после случайного перемешивания одной из величин.

Если $X$ и $Y$ независимы, то $f_{XY}(x,y) = f_X(x)\,f_Y(y)$, и все эти показатели должны быть около нуля.

In [ ]:
pair_tests(T, I)

In [ ]:
pair_tests(T, A)

**Вывод.** Время на сайте и интернет-активность **зависимы**: $r = 0.52$, $\rho = 0.51$, $\chi^2 = 567$ ($p \approx 10^{-110}$), взаимная информация 0.43 против 0.008 после перемешивания. Чем больше человек сидит на сайте, тем больше он вообще сидит в интернете. Время на сайте и возраст тоже зависимы ($r = -0.33$, $\chi^2 = 209$, взаимная информация 0.20): более старшие проводят на сайте меньше времени. Независимыми эти величины считать нельзя.

#### Состоятельность оценок
Оценка состоятельна, если при росте объёма выборки $n$ она приближается к истинному значению параметра. Проверяем эмпирически: берём из совокупности 500 выборок для каждого $n$ (от 10 до 5000, с возвращением), считаем по каждой среднее, дисперсию и медиану и смотрим, как уменьшается их разброс. Для среднего теория даёт $\mathrm{sd}(\bar X) = \sigma / \sqrt{n}$ [15].

In [ ]:
consistency(T)

**Вывод.** Разброс выборочного среднего убывает практически по кривой $\sigma/\sqrt{n}$ (например, при $n=100$: 1.53 в опыте и 1.58 по теории; при $n=5000$: 0.226 и 0.224). Среднее выборочных средних не смещено (смещение близко к 0). Разброс оценки дисперсии падает с 90 до 3.2, медианы - с 7.5 до 0.36, то есть эти оценки тоже состоятельны. Медиана при этом разбросана больше среднего (при $n = 100$: 2.37 против 1.53).

#### Выборка из генеральной совокупности: с возвращением и без
Генеральной совокупностью считаем все 1000 строк. Берём выборки по $n = 100$ **и повторяем 2000 раз** (одна выборка, как в начале, не годится: оценка по одной выборке слишком случайна). Теория: без возвращения разброс среднего меньше в $\sqrt{\frac{N-n}{N-1}}$ раз (поправка на конечную совокупность [13]); здесь $\sqrt{900/999} \approx 0.949$.

In [ ]:
sampling_compare(T)

**Вывод.** Среднее выборочных средних в обоих случаях очень близко к среднему совокупности (65.02 против 65.00), то есть оба способа дают несмещённую оценку. Различие в разбросе: без возвращения 1.49 (теория 1.50), с возвращением 1.61 (теория 1.58), так что выборка без возвращения даёт чуть более точную оценку, как и предсказывает теория. Средняя оценка стандартного отклонения (15.83 и 15.81) немного ниже истинной (15.85) - это известное небольшое занижение у малых выборок. Для такой большой совокупности ($n$ = 10% от $N$) разница между способами небольшая; с ростом доли выборки в совокупности она бы росла.

<a id="rv1-hyp"></a>
#### Проверка гипотезы
**$H_0$:** время на сайте у кликнувших и не кликнувших на рекламу распределено одинаково (в сдвиге значений разницы нет).
**$H_1$:** распределения различаются.

Берём критерий Манна-Уитни [9]: он не требует нормальности, а у нас её нет. Для контроля - критерий Уэлча. Силу эффекта показывает вероятность того, что случайный кликнувший провёл на сайте меньше, чем случайный не кликнувший.

In [ ]:
click_test(T)

**Вывод.** $H_0$ отвергаем: $p \approx 10^{-122}$ (Манн-Уитни), $p \approx 10^{-168}$ (Уэлч). Медиана у кликнувших 51.5 мин, у не кликнувших 77.7 мин. Вероятность того, что случайный кликнувший провёл на сайте меньше времени, равна 0.93. Это очень сильный эффект.

**Как это интерпретировать.** Это связь, а не причина: нельзя сказать, что мало времени на сайте «заставляет» кликать. И главное: в реальных данных пересечение групп обычно намного больше, а такое чистое разделение (93%) выглядит нереалистично и ещё раз подтверждает, что набор синтетический.

<a id="rv2"></a>
## Random Variable 02 - интернет-активность (`Daily Internet Usage`)

<a id="rv2-desc"></a>
### Description
**Природа события.** Сколько времени (мин) пользователь проводит в интернете за день. Величина непрерывная, неотрицательная, по смыслу близка к первой величине (тоже время), поэтому их независимость мы проверяли в разделе про первую величину (там же и пара «время на сайте - возраст»). Здесь проверяем случайность тем же способом: регрессия на остальные признаки, автокорреляция, критерий серий.

In [ ]:
randomness_check(I)

**Вывод.** По остальным признакам формула даёт $R^2 = 0.34$ (0.63 вместе с `Clicked on Ad`), остатки широкие. Автокорреляция 0.02, критерий серий $p = 0.66$. Значение детерминированной формулой не вычисляется и порядка в строках нет, то есть это случайная величина (с той же оговоркой про синтетические данные).

<a id="rv2-eda"></a>
### EDA & Distribution

In [ ]:
eda(I)

In [ ]:
by_group(I)

In [ ]:
normality(I)

In [ ]:
normalization(I)

**Выводы по EDA.** Среднее 180.0, медиана 183.1, стандартное отклонение 43.9, значения от 104.8 до 270.0. Асимметрия почти нулевая ($-0.03$), то есть распределение симметричное, но эксцесс $-1.27$: оно очень плоское (у равномерного распределения эксцесс равен $-1.2$). Нормальность отвергается всеми тестами. По группам картина разная: у не кликнувших среднее 214.5 ($\sigma = 23.9$), у кликнувших 145.5 ($\sigma = 30.0$), причём у кликнувших группа скошена вправо (асимметрия 1.24, эксцесс 1.82). Нормализация (z-оценка) работает как в первой величине: среднее 0, станд. откл. 1, форма не меняется.

In [ ]:
mixture_fit(I)

#### Математическая база распределения
Математика та же, что и для времени на сайте: смесь двух нормальных $f(x) = w_1 f_1(x) + w_2 f_2(x)$ с весами 0.5 и 0.5 (формулы среднего и дисперсии смеси см. выше). Среднее и дисперсия смеси совпадают с выборочными (180.00 и 1927.0 против 1925.5). По KS смесь лучше одной нормальной ($p = 0.028$ против $p \approx 0$), но на уровне 5% всё же отвергается. Причина видна по группам: у кликнувших хвост справа длиннее, а нормальные компоненты симметричны. Значит, смесь двух нормальных здесь - приближение, а не точная модель (можно было бы взять для этой группы гамма-распределение, но мы это оставляем).

<a id="rv2-ind"></a>
### Independence & Estimations
Пара «время на сайте - интернет-активность» разобрана выше. Здесь проверяем интернет-активность и возраст.

In [ ]:
pair_tests(I, A)

**Вывод.** Интернет-активность и возраст зависимы: $r = -0.37$, $\rho = -0.37$, $\chi^2 = 223$, взаимная информация 0.21 против 0.006 после перемешивания. Более старшие пользователи в среднем меньше сидят в интернете.

#### Состоятельность оценок и выборки
Те же опыты, что и для первой величины.

In [ ]:
consistency(I)

In [ ]:
sampling_compare(I)

**Вывод.** Разброс среднего убывает примерно как $\sigma/\sqrt{n}$ (от 13.8 при $n=10$ до 0.64 при $n=5000$), разбросы дисперсии и медианы тоже падают, смещение среднего около 0, то есть оценки состоятельны. В опыте 2000 выборок по 100: без возвращения среднее 179.99 и разброс 4.16 (теория 4.16), с возвращением 179.89 и разброс 4.32 (теория 4.39). Вывод тот же: оба способа несмещены, без возвращения разброс чуть меньше.

#### Проверка гипотезы
**$H_0$:** интернет-активность у кликнувших и не кликнувших одинакова; **$H_1$:** различается. Критерий Манна-Уитни.

In [ ]:
click_test(I)

**Вывод.** $H_0$ отвергаем ($p \approx 10^{-133}$). Медиана у кликнувших 138.8 мин, у не кликнувших 216.4 мин, вероятность того, что кликнувший сидел в интернете меньше, равна 0.95. Этот эффект ещё сильнее, чем у времени на сайте. Интерпретация та же: связь есть, причинность не доказана, а чистота разделения говорит о синтетических данных.

<a id="rv3"></a>
## Random Variable 03 - возраст (`Age`)

<a id="rv3-desc"></a>
### Description
**Природа события.** Возраст пользователя в годах. Это **дискретная** величина: целые значения от 19 до 61 (43 значения). Возраст конкретного человека, конечно, известен, но возраст случайно попавшего на сайт пользователя заранее не определён: он зависит от того, кто именно пришёл. Проверяем случайность так же, как и раньше.

In [ ]:
randomness_check(A)

**Вывод.** По остальным признакам формула объясняет всего $R^2 = 0.16$ (0.25 с `Clicked on Ad`), порядка по строкам нет (автокорреляция $-0.03$, критерий серий $p = 0.75$). Возраст формулой не вычисляется, это случайная величина.

<a id="rv3-eda"></a>
### EDA & Distribution

In [ ]:
eda(A)

In [ ]:
by_group(A)

In [ ]:
normality(A)

In [ ]:
normalization(A)

In [ ]:
# подгонка теоретических законов: правдоподобие для целых значений P(k) = F(k+0.5) - F(k-0.5)
ages = df[A].to_numpy()
vals, cnt = np.unique(ages, return_counts=True)
rows = []
for name, dist in [('нормальное', stats.norm), ('гамма (со сдвигом)', stats.gamma),
                   ('логнормальное (со сдвигом)', stats.lognorm), ('скошенное нормальное', stats.skewnorm)]:
    par = dist.fit(ages)
    pk = dist.cdf(vals + 0.5, *par) - dist.cdf(vals - 0.5, *par)
    ll = (cnt * np.log(pk)).sum()
    exp = pk / pk.sum() * len(ages)
    # объединяем соседние значения, пока ожидаемая частота < 5
    o_b, e_b, so, se = [], [], 0, 0
    for o, e in zip(cnt, exp):
        so += o; se += e
        if se >= 5: o_b.append(so); e_b.append(se); so = se = 0
    o_b[-1] += so; e_b[-1] += se
    e_b = np.array(e_b) * sum(o_b) / sum(e_b)
    chi2 = ((np.array(o_b) - e_b) ** 2 / e_b).sum(); dof = len(o_b) - 1 - len(par)
    rows.append({'закон': name, 'параметры': tuple(round(p, 2) for p in par), 'AIC': 2 * len(par) - 2 * ll,
                 'chi2': chi2, 'степ. свободы': dof, 'p chi2': 1 - stats.chi2.cdf(chi2, dof)})
fits = pd.DataFrame(rows).set_index('закон')
fits.round(4)

In [ ]:
grid = np.linspace(17, 63, 300)
par = stats.skewnorm.fit(ages)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(vals, cnt / len(ages), color=C1, alpha=0.5, label='данные (доля)')
ax.plot(grid, stats.skewnorm.pdf(grid, *par), color=C2, lw=2, label='скошенное нормальное')
ax.plot(grid, stats.norm.pdf(grid, ages.mean(), ages.std(ddof=1)), color='gray', ls='--', label='нормальное')
ax.set(title='Возраст: данные и подогнанные законы', xlabel='Возраст, лет', ylabel='Доля / плотность'); ax.legend()
plt.tight_layout(); plt.show()

**Выводы по EDA.** Среднее 36.0, медиана 35, стандартное отклонение 8.79, значения от 19 до 61. Асимметрия $+0.48$ (хвост справа: взрослых много, пожилых мало), эксцесс $-0.40$. Нормальность отвергается. По группам: не кликнувшие в среднем моложе (31.7 года), кликнувшие старше (40.3 года). Нормализация работает как раньше (среднее 0, станд. откл. 1, форма сохраняется).

#### Математическая база распределения
Нормальное не подходит ($\chi^2 = 89.4$ при 35 степенях свободы, $p \approx 0$), поэтому пробуем законы с асимметрией:
- **гамма-распределение со сдвигом** $c$: $f(x) = \dfrac{(x-c)^{k-1} e^{-(x-c)/\theta}}{\Gamma(k)\,\theta^k}$, $x > c$; среднее $c + k\theta$, асимметрия $2/\sqrt{k}$ (всегда положительная);
- **логнормальное со сдвигом**: логарифм величины нормален;
- **скошенное нормальное** [16]: $f(x) = \dfrac{2}{\omega}\,\varphi\!\left(\dfrac{x-\xi}{\omega}\right)\Phi\!\left(\alpha\,\dfrac{x-\xi}{\omega}\right)$, где $\varphi$ и $\Phi$ - плотность и функция распределения $N(0,1)$, $\xi$ - положение, $\omega$ - масштаб, $\alpha$ - параметр скоса (при $\alpha = 0$ получаем обычное нормальное).

Поскольку возраст целый, вероятность каждого значения считаем как $P(k) = F(k + 0.5) - F(k - 0.5)$ и сравниваем законы по критерию Акаике $\mathrm{AIC} = 2m - 2\ln L$ (чем меньше, тем лучше; $m$ - число параметров) [17] и по $\chi^2$ с объединением редких значений (ожидаемая частота не меньше 5).

**Вывод по подгонке.** Лучший по обоим показателям - скошенное нормальное (AIC 7114.5, $p_{\chi^2} = 0.34$), чуть хуже гамма (7118.8; 0.23) и логнормальное (7126.1; 0.14), все три не отвергаются. Нормальное явно хуже (AIC 7187.1). Параметры скошенного нормального: $\alpha \approx 3.7$, $\xi \approx 25.1$, $\omega \approx 14.0$.

<a id="rv3-ind"></a>
### Independence & Estimations
Пары с возрастом: время на сайте - возраст и интернет-активность - возраст разобраны в разделах выше, итоговая таблица по всем трём парам внизу этого раздела.

In [ ]:
consistency(A)

In [ ]:
sampling_compare(A)

**Вывод.** Разброс среднего убывает как $\sigma/\sqrt{n}$ (при $n = 100$: 0.86 против 0.88 по теории; при $n = 5000$: 0.126 и 0.124), смещение около 0, оценки состоятельны. Исключение - медиана: при $n = 5000$ её разброс (0.077) оказался заметно меньше теоретического. Это особенность дискретной величины: медиана выпадает на целое число и почти всегда выдаёт одно и то же значение (35) или соседнее, поэтому разброс при больших $n$ ведёт себя «скачками». Для выборок по 100 (2000 повторов) без возвращения среднее 36.02, разброс 0.833 (теория 0.834), с возвращением 36.03 и 0.877 (теория 0.878). Вывод тот же, что и у предыдущих величин.

#### Проверка гипотезы
**$H_0$:** возраст у кликнувших и не кликнувших одинаков; **$H_1$:** различается. Критерий Манна-Уитни.

In [ ]:
click_test(A)

**Вывод.** $H_0$ отвергаем ($p \approx 10^{-53}$). Кликнувшие старше: медиана 40 лет против 31. Вероятность того, что случайный кликнувший моложе случайного не кликнувшего, равна 0.22 (то есть с вероятностью 0.78 он старше). Эффект заметный, но слабее, чем для времени на сайте и интернет-активности: возрастные группы пересекаются больше.

#### Сводная таблица независимости по всем парам

In [ ]:
# итоговая таблица по всем трём парам
pairs = [(T, I), (T, A), (I, A)]
summary = pd.concat([pair_tests(a, b, plot=False) for a, b in pairs], axis=1)
summary.round(4)

**Вывод по независимости.** Все три пары зависимы (все $p$ практически нулевые, взаимная информация в 28-57 раз больше, чем после перемешивания). Самая сильная связь между временем на сайте и интернет-активностью ($r = 0.52$), с возрастом обе величины связаны отрицательно ($r = -0.33$ и $-0.37$). Практический смысл: три величины нельзя генерировать независимо и считать, что получится «настоящая» аудитория, - связи теряются. Мы это учитываем в sprint 2 как ограничение.

<a id="s1-resume"></a>
## Lab resume (sprint 1)
**Общие выводы.**
1. Все три величины - случайные: формулой по остальным признакам не вычисляются ($R^2$ от 0.16 до 0.34 без целевого признака) и не имеют порядка в строках.
2. Ни одна величина не нормальна. Время на сайте и интернет-активность хорошо описываются смесью двух нормальных (по группам кликов): их плоская форма (эксцесс $-1.1$ и $-1.3$) вызвана тем, что в данных смешаны две группы с разным поведением. Возраст лучше всего описывает скошенное нормальное распределение.
3. Все три величины попарно зависимы. Время на сайте и интернет-активность связаны сильнее всего ($r = 0.52$), с возрастом связь отрицательная.
4. Оценки среднего, дисперсии и медианы состоятельны. Выборки с возвращением и без одинаково несмещены, без возвращения разброс меньше примерно на 5% (множитель 0.949), как предсказывает теория.
5. Гипотезы о различии величин между кликнувшими и нет подтверждены для всех трёх величин. Кликнувшие проводят на сайте и в интернете меньше времени и в среднем старше.
6. Набор данных очень вероятно синтетический (500/500, круглые средние, 93% разделимость по времени на сайте), поэтому выводы относятся к этому набору и не переносятся на реальных пользователей без проверки.

Критерии оценивания, которые мы считаем выполненными, в таблице ниже (значения `coeff`: 1 - выполнено, 0.5 - выполнено частично, 0 - не выполнено).

### cat_mark_table
| cat_id | notebook_link | coeff | comment |
|--------|---------------|-------|---------|
| Случайность величины (×3) | [RV1](#rv1-desc), [RV2](#rv2-desc), [RV3](#rv3-desc) | 1 | регрессия на остальные признаки, автокорреляция, критерий серий; оговорка про синтетику |
| Независимость | [RV1](#rv1-ind), [RV2](#rv2-ind), [RV3](#rv3-ind) | 1 | три пары; Пирсон, Спирмен, $\chi^2$, взаимная информация с контролем перемешиванием |
| Состоятельность оценок | [RV1](#rv1-ind), [RV2](#rv2-ind), [RV3](#rv3-ind) | 1 | среднее, дисперсия, медиана; сравнение с $\sigma/\sqrt{n}$ |
| EDA и нормализация | [RV1](#rv1-eda), [RV2](#rv2-eda), [RV3](#rv3-eda) | 1 | z-оценка и min-max, объяснено, что форма не меняется |
| Выборка: с возвращением и без | [RV1](#rv1-ind), [RV2](#rv2-ind), [RV3](#rv3-ind) | 1 | 2000 повторов, сравнение с теорией и поправкой на конечную совокупность |
| Математическая база распределения | [RV1](#rv1-eda), [RV2](#rv2-eda), [RV3](#rv3-eda) | 1 | смесь нормальных (двух величин), скошенное нормальное / гамма / логнормальное (возраст) |
| Проверка гипотез | [RV1](#rv1-hyp), [RV2](#rv2-ind), [RV3](#rv3-ind) | 1 | Манн-Уитни и Уэлч, размер эффекта; три гипотезы |
| Приведение к стандартному нормальному | [RV1](#rv1-eda) | 0.5 | z-нормализация описана и проведена, но данные не нормальные |
| Каждое дополнительное распределение | [RV3](#rv3-eda) | 1 | 4 закона для возраста, сравнение по AIC и $\chi^2$ |
| Дополнительный датасет | - | 0 | не делали |
| Литература | [sprint 2](#lit) | 1 | общий список в конце ноутбука |

<a id="sprint2"></a>
# Lab01 - sprint 2
Строим датчики случайных чисел для двух величин: возраста (дискретная) и времени на сайте (непрерывная). Источником равномерных чисел служит **собственная реализация вихря Мерсенна** (файл `generators.py`), готовый `numpy` используем только как эталон для проверки. Автотесты лежат в `test_generators.py` (запуск: `pytest`).

## Оглавление (sprint 2)
- [Базовый датчик: вихрь Мерсенна](#mt)
- [Random 01 - возраст](#r1): [описание](#r1-desc) · [математика](#r1-math) · [код](#r1-code) · [тесты](#r1-tests) · [итоги](#r1-resume)
- [Random 02 - время на сайте](#r2): [описание](#r2-desc) · [математика](#r2-math) · [код](#r2-code) · [тесты](#r2-tests) · [итоги](#r2-resume)
- [Lab resume (sprint 2)](#s2-resume) · [Литература](#lit)


<a id="mt"></a>
## Базовый датчик: вихрь Мерсенна (MT19937)
Все остальные датчики берут равномерные числа $U \sim U[0,1)$. Мы пишем их источник сами.

**Как устроен.** Внутреннее состояние - 624 слова по 32 бита. Из начального числа (seed) оно заполняется рекуррентно: $x_i = 1812433253\,(x_{i-1} \oplus (x_{i-1} \gg 30)) + i \pmod{2^{32}}$. Дальше состояние обновляется линейной рекуррентностью над полем $\mathbb{F}_2$ [1]:
$$x_{k+624} = x_{k+397} \oplus \big( (x_k^{\,u} \,|\, x_{k+1}^{\,l})\, A \big),$$
где $x^u$ - старший бит слова, $x^l$ - его младшие 31 бит, $A$ - фиксированная матрица. Каждое выданное слово дополнительно «перемешивается» (tempering): четыре побитовых сдвига и маскирования, чтобы улучшить равномерность по старшим разрядам.

**Свойства.** Период $2^{19937} - 1$, равнораспределённость в 623 измерениях по 32 бита [1]. Из двух слов получаем число с 53 битами точности: $U = (a \cdot 2^{26} + b)/2^{53}$, где $a$ - 27 старших бит одного слова, $b$ - 26 старших бит следующего.

**Ограничения.** Это линейный генератор: он не подходит для криптографии (по 624 подряд идущим выходам состояние восстанавливается) и не проходит некоторые строгие тесты на линейную сложность [14]. Для статистического моделирования этого более чем достаточно.

**Проверка.** Раз `numpy.random.RandomState` использует тот же алгоритм и инициализацию, наши числа должны совпасть с его выходом бит в бит.

In [ ]:
import time
from generators import MT19937, discrete_inverse, alias_sample, normal_polar, normal_mixture
import test_generators

mt = MT19937(42)
print('первые 5 чисел (моё):   ', mt.random(5))
print('первые 5 чисел (numpy): ', np.random.RandomState(42).random_sample(5))
print('seed=5489, первое uint32:', MT19937(5489).uint32(), '(эталон из статьи: 3499211612)')

u = MT19937(1).random(100000)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(u, bins=20, stat='density', color=C1, ax=ax[0]); ax[0].axhline(1, color=C2, ls='--')
ax[0].set(title='Равномерность U(0,1)', xlabel='u', ylabel='Плотность')
ax[1].scatter(u[:3000], u[1:3001], s=2, color=C1)
ax[1].set(title='Пары соседних чисел', xlabel='u_i', ylabel='u_(i+1)')
plt.tight_layout(); plt.show()

**Вывод.** Наши первые числа совпадают с `numpy` до последнего знака, а первое 32-битное слово при seed = 5489 равно эталонному значению 3499211612 из реализации авторов. Гистограмма близка к равномерной, на диаграмме пар соседних чисел нет структуры. Дальше проверки на уровне тестов (равномерность $\chi^2$, автокорреляция) идут в `test_generators.py`.

<a id="r1"></a>
## Random 01 - возраст (дискретная величина)
<a id="r1-desc"></a>
### Description

Возраст - дискретная величина с 43 значениями (19-61). Мы моделируем её по эмпирическому закону: вероятность каждого возраста равна его доле в данных, $p_k = n_k / N$. Это модель «как в данных»; она не умеет выдавать значения, которых в данных нет (например, 18 или 62 года), и в этом ограничение: по сути это взвешенный выбор из уже имеющихся значений. Для сравнения реализуем **два разных алгоритма**: метод обратной функции и метод псевдонимов.

<a id="r1-math"></a>
### math desc
**Метод обратной функции (дискретный).** Значения $x_1 < x_2 < \dots < x_m$, вероятности $p_k$, функция распределения $F(x_k) = \sum_{j \le k} p_j$. Генерируем $U \sim U(0,1)$ и берём $X = x_k$, где $k$ - наименьший номер с $F(x_k) \ge U$. Почему это работает:
$$P(X = x_k) = P\big(F(x_{k-1}) < U \le F(x_k)\big) = F(x_k) - F(x_{k-1}) = p_k.$$
В коде: накопленные суммы и бинарный поиск (`searchsorted`), сложность $O(\log m)$ на число [3].

**Метод псевдонимов (Уокера-Воуза).** Идея: разложить распределение на $m$ равных столбцов, в каждом из которых не более двух значений: «своё» с вероятностью $q_k$ и «псевдоним» $a_k$ с вероятностью $1 - q_k$. Генерация: $K = \lfloor m U_1 \rfloor$, затем если $U_2 < q_K$, то $X = x_K$, иначе $X = x_{a_K}$. Тогда
$$P(X = x_j) = \frac{1}{m}\Big(q_j + \sum_{k:\, a_k = j} (1 - q_k)\Big) = p_j,$$
что мы и проверяем автотестом `test_alias_tables_sum_to_probs`. Таблицу строит алгоритм Воуза за $O(m)$ [5, 6], после чего каждое число дают за $O(1)$ (два равномерных числа). Метод выигрывает, когда значений очень много.

<a id="r1-code"></a>
### code

In [ ]:
ages = df[A].to_numpy()
vals, counts = np.unique(ages, return_counts=True)
pmf = counts / counts.sum()
N_GEN = 100000

t0 = time.perf_counter()
age_inv = discrete_inverse(vals, pmf, MT19937(2024).random(N_GEN))
t_inv = time.perf_counter() - t0
t0 = time.perf_counter()
age_alias = alias_sample(vals, pmf, MT19937(2025), N_GEN)
t_alias = time.perf_counter() - t0
print(f'обратная функция: {t_inv:.2f} c, псевдонимы: {t_alias:.2f} c (источник чисел - мой вихрь Мерсенна)')
print('первые 15:', age_inv[:15], age_alias[:15])

<a id="r1-tests"></a>
### tests
Сравниваем сгенерированные 100000 значений с реальными. Показатели: основные статистики, расстояние Вассерштейна и двухвыборочный критерий Колмогорова-Смирнова (KS).

In [ ]:
def compare(real, gen):
    ks = stats.ks_2samp(real, gen)
    return pd.Series({'среднее': np.mean(gen), 'медиана': np.median(gen), 'станд. откл.': np.std(gen, ddof=1),
                      'асимметрия': stats.skew(gen), 'эксцесс': stats.kurtosis(gen), 'мин': np.min(gen), 'макс': np.max(gen),
                      'расстояние Вассерштейна': stats.wasserstein_distance(real, gen),
                      'KS статистика': ks.statistic, 'KS p-value': ks.pvalue})

res_age = pd.DataFrame({'реальные данные': pd.Series({'среднее': ages.mean(), 'медиана': np.median(ages), 'станд. откл.': ages.std(ddof=1),
                         'асимметрия': stats.skew(ages), 'эксцесс': stats.kurtosis(ages), 'мин': ages.min(), 'макс': ages.max()}),
                        'обратная функция': compare(ages, age_inv), 'псевдонимы': compare(ages, age_alias)})
res_age.round(4)

In [ ]:
# проверка алгоритма: частоты по каждому из 43 возрастов против вероятностей pmf
rows = {}
for name, g in [('обратная функция', age_inv), ('псевдонимы', age_alias)]:
    obs = np.array([(g == v).sum() for v in vals])
    c = stats.chisquare(obs, pmf * len(g))
    rows[name] = {'chi2': c.statistic, 'степ. свободы': len(vals) - 1, 'p-value': c.pvalue}
display(pd.DataFrame(rows).T.round(4))

fig, ax = plt.subplots(figsize=(11, 4))
w = 0.28
ax.bar(vals - w, pmf, w, color='gray', label='данные')
ax.bar(vals, np.array([(age_inv == v).mean() for v in vals]), w, color=C1, label='обратная функция')
ax.bar(vals + w, np.array([(age_alias == v).mean() for v in vals]), w, color=C2, label='псевдонимы')
ax.set(title='Доли каждого возраста', xlabel='Возраст, лет', ylabel='Доля'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# автотесты (те же, что запускает pytest из test_generators.py)
names = [n for n in dir(test_generators) if n.startswith('test_')]
for n in names:
    getattr(test_generators, n)()
    print('OK ', n)
print(f'Пройдено тестов: {len(names)} из {len(names)}')

<a id="r1-resume"></a>
### resume
Оба алгоритма воспроизводят данные: среднее 36.03 и 36.04 (в данных 36.01), стандартное отклонение 8.78 и 8.81 (8.79), асимметрия 0.486 и 0.476 (0.478), эксцесс $-0.39$ и $-0.42$ ($-0.40$), медиана 35 и границы 19-61 совпадают. KS: $p \approx 1$ в обоих случаях. Частоты по каждому из 43 возрастов совпадают с вероятностями (критерий $\chi^2$: $p = 0.65$ и $0.37$). Все автотесты пройдены.

**Замечания.** (1) Сравнение с данными не очень строгое: генератор по построению воспроизводит их распределение, поэтому $p \approx 1$ ожидаемо. Больше информации в проверке $\chi^2$, которая показывает, что алгоритмы правильно реализуют заданные вероятности. (2) В нашей реализации метод псевдонимов медленнее (0.62 с против 0.20 с на 100000 чисел), хотя в теории он быстрее: векторизованный бинарный поиск из `numpy` быстрее цикла на Python, а число значений небольшое (43). Преимущество псевдонимов проявляется на гораздо больших таблицах.

<a id="r2"></a>
## Random 02 - время на сайте (непрерывная величина)
<a id="r2-desc"></a>
### Description

Время на сайте непрерывно, а в sprint 1 мы выяснили, что это смесь двух групп (кликнувшие и нет) с разным поведением. Поэтому моделируем его как смесь нормальных распределений, а нормальные числа получаем полярным методом на основе нашего вихря Мерсенна. Работа шла в два шага (две версии), второй возник из-за проблемы, которую показала проверка первого.

<a id="r2-math"></a>
### math desc
**Метод суперпозиции.** Плотность $f = \sum_k w_k f_k$. Сначала выбираем номер компоненты $K$ с вероятностями $P(K = k) = w_k$ (метод обратной функции для дискретной величины), затем генерируем $X$ из $f_K$. По формуле полной вероятности
$$P(X \le x) = \sum_k P(K = k)\,P(X \le x \mid K = k) = \sum_k w_k F_k(x),$$
то есть $X$ имеет нужное распределение [2, 3].

**Полярный метод Марсальи для $N(0,1)$.** Берём $V_1, V_2 \sim U(-1, 1)$, считаем $S = V_1^2 + V_2^2$ и принимаем пару, только если $0 < S < 1$ (точка попала в круг; вероятность принятия $\pi/4 \approx 0.785$). Тогда
$$Z_1 = V_1 \sqrt{\frac{-2 \ln S}{S}}, \qquad Z_2 = V_2 \sqrt{\frac{-2 \ln S}{S}}$$
- независимые $N(0,1)$ [4]. Почему: для точки, равномерной в круге, в полярных координатах $S = R^2 \sim U(0,1)$ и угол $\Theta$ равномерен и от $S$ не зависит, а $V_1/\sqrt{S} = \cos\Theta$. Значит, $Z_1 = \sqrt{-2\ln S}\cos\Theta$, а это в точности формула Бокса-Мюллера [7], только без вычисления синуса и косинуса.

**Усечение методом Неймана (отбраковка).** Если генерировать из плотности $f$ и выбрасывать значения вне $[a, b]$, оставшиеся имеют плотность
$$f_{[a,b]}(x) = \frac{f(x)}{F(b) - F(a)}, \quad x \in [a, b],$$
а доля принятых равна $F(b) - F(a)$ [18, 3]. Параметры $\mu, \sigma$ усечённого нормального находим методом максимального правдоподобия, максимизируя $\sum_i \ln\dfrac{\varphi\big((x_i - \mu)/\sigma\big)/\sigma}{\Phi(b') - \Phi(a')}$, где $a' = (a-\mu)/\sigma$, $b' = (b-\mu)/\sigma$ (обычные выборочные среднее и стандартное отклонение здесь не годятся: по усечённым данным они смещены).

<a id="r2-code"></a>
### code
**Версия 1:** смесь двух обычных нормальных с параметрами, равными выборочным средним и стандартным отклонениям по группам.

In [ ]:
g = df['Clicked on Ad'].to_numpy(); x_t = df[T].to_numpy(float)
w_t = [np.mean(g == k) for k in (0, 1)]
m_t = [x_t[g == k].mean() for k in (0, 1)]
s_t = [x_t[g == k].std(ddof=1) for k in (0, 1)]
print('Версия 1 (смесь нормальных): веса', [float(v) for v in w_t], '| средние', np.round(m_t, 2), '| станд. откл.', np.round(s_t, 2))
time_v1 = normal_mixture(MT19937(7), N_GEN, w_t, m_t, s_t)
print('Диапазон: данные', x_t.min(), '-', x_t.max(), '| версия 1', round(time_v1.min(), 1), '-', round(time_v1.max(), 1))

**Проблема версии 1.** Датчик выдаёт значения от 1 до 115 минут, хотя в данных они от 32.6 до 91.4: нормальные хвосты бесконечны, а у реальной величины границы жёсткие. Из-за этого у версии 1 эксцесс получается $-0.48$ вместо $-1.10$ в данных, а KS-тест почти на границе отвержения ($p = 0.053$).

**Версия 2:** каждая компонента усечена на диапазон своей группы ($[48.2; 91.4]$ и $[32.6; 91.4]$), параметры нормальных до усечения оценены методом максимального правдоподобия, числа получены методом Неймана. Сравнение версий:

In [ ]:
from generators import truncated_mixture, fit_truncated_normal
params = []
for k in (0, 1):
    xk = x_t[g == k]; lo, hi = xk.min(), xk.max()
    mu, sd = fit_truncated_normal(xk, lo, hi)
    params.append((mu, sd, lo, hi))
    print(f'группа {k}: mu = {mu:.2f}, sigma = {sd:.2f}, границы [{lo}, {hi}]')

t0 = time.perf_counter()
time_v2 = truncated_mixture(MT19937(3), N_GEN, w_t, params)
print(f'{N_GEN} чисел за {time.perf_counter() - t0:.2f} c')

def stat_col(v):
    return pd.Series({'среднее': v.mean(), 'медиана': np.median(v), 'станд. откл.': v.std(ddof=1), 'асимметрия': stats.skew(v),
                      'эксцесс': stats.kurtosis(v), 'мин': v.min(), 'макс': v.max()})
res_t = pd.DataFrame({'реальные данные': stat_col(x_t), 'версия 1: смесь': compare(x_t, time_v1), 'версия 2: усечённая смесь': compare(x_t, time_v2)})
res_t.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x_t, bins=30, stat='density', color='gray', alpha=0.45, label='данные', ax=ax[0])
ax[0].hist(time_v1, bins=60, density=True, histtype='step', color=C1, lw=1.5, label='версия 1')
ax[0].hist(time_v2, bins=60, density=True, histtype='step', color=C2, lw=1.5, label='версия 2')
ax[0].set(title='Время на сайте: данные и датчики', xlabel=LABELS[T], ylabel='Плотность'); ax[0].legend()
q = np.linspace(0.01, 0.99, 99)
ax[1].plot(np.quantile(x_t, q), np.quantile(time_v1, q), 'o', ms=3, color=C1, label='версия 1')
ax[1].plot(np.quantile(x_t, q), np.quantile(time_v2, q), 'o', ms=3, color=C2, label='версия 2')
lim = [x_t.min(), x_t.max()]; ax[1].plot(lim, lim, color='gray')
ax[1].set(title='QQ-график: данные против датчика', xlabel='Квантили данных, мин', ylabel='Квантили датчика, мин'); ax[1].legend()
plt.tight_layout(); plt.show()

<a id="r2-tests"></a>
### tests
Проверяем нормальную часть датчика отдельно (и на нескольких seed, чтобы не опираться на один случайный $p$-value), затем запускаем все автотесты ещё раз.

In [ ]:
z = normal_polar(MT19937(11), 100000)
print('N(0,1), 100000 чисел: среднее', round(z.mean(), 4), '| std', round(z.std(), 4), '| асимметрия', round(stats.skew(z), 4),
      '| эксцесс', round(stats.kurtosis(z), 4), '| KS p =', round(stats.kstest(z, 'norm').pvalue, 4))
# один p-value мало что значит: повторяем с 30 разными seed и смотрим, как часто p < 0.05 (ожидаем ~5%)
pv = [stats.kstest(normal_polar(MT19937(s), 5000), 'norm').pvalue for s in range(100, 130)]
print(f'30 серий по 5000 чисел: p < 0.05 в {sum(p < 0.05 for p in pv)} из 30, медиана p = {np.median(pv):.2f}')
names = [n for n in dir(test_generators) if n.startswith('test_')]
for n in names:
    getattr(test_generators, n)()
print(f'Автотесты после всех доработок: {len(names)} из {len(names)} пройдено')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<a id="r2-resume"></a>
### resume
Итоговая таблица показывает, что **версия 2 заметно лучше**: среднее 65.00 (в данных 65.00), стандартное отклонение 15.85 (15.85), асимметрия $-0.365$ ($-0.371$), эксцесс $-1.04$ ($-1.10$), диапазон 32.6-91.4 совпадает с данными (у версии 1 он 1.0-114.7). Двухвыборочный KS: $p = 0.76$ у версии 2 против $0.053$ у версии 1; расстояние Вассерштейна 0.32 мин против 1.19 мин. Полярный метод даёт нормальные числа: среднее 0.007, станд. откл. 0.999, эксцесс 0.008; одиночный $p$ KS у 100 тысяч чисел оказался 0.046, но на 30 разных seed ниже 0.05 было 3 из 30, что в рамках ожидаемых 5% случайных срабатываний. Все 13 автотестов проходят.

**Ограничения.** Эксцесс версии 2 всё же чуть выше ($-1.04$ против $-1.10$), форма внутри групп приближённая (в группах есть скос). Главное: наш датчик генерирует время на сайте **независимо** от возраста и интернет-активности, а в данных эти величины связаны ($r = -0.33$ и $0.52$). Для совместного датчика нужно моделировать вместе.

<a id="s2-resume"></a>
## Lab resume (sprint 2)
**Что сделано.** Собственный вихрь Мерсенна (совпадает с `numpy` бит в бит), два датчика для возраста (обратная функция и метод псевдонимов), датчик для времени на сайте в двух версиях (смесь нормальных на полярном методе, затем усечённая смесь с отбраковкой Неймана), 13 автотестов. Метрики реальных данных и датчиков совпадают (таблицы выше).

**Чего не сделано.** Датчики независимы друг от друга, поэтому связи между величинами не воспроизводятся; не делался отдельный датасет и интерактивность.

### cat_mark_table
| cat_id | notebook_link | coeff | comment |
|--------|---------------|-------|---------|
| Метрики данных и датчика совпадают | [Random 01](#r1-tests), [Random 02](#r2-code) | 1 | таблицы метрик, KS, Вассерштейн, $\chi^2$ |
| Метод обратной функции | [Random 01](#r1-math) | 1 | вывод в math desc, реализация и тесты |
| Другой алгоритм датчика (метод псевдонимов) | [Random 01](#r1-math) | 1 | Уокер-Воуз, сравнение по времени |
| Нормальные числа: полярный метод | [Random 02](#r2-math) | 1 | вывод связи с Боксом-Мюллером |
| Метод суперпозиции | [Random 02](#r2-math) | 1 | смесь двух компонент |
| Метод Неймана (отбраковка) | [Random 02](#r2-math) | 1 | усечённые компоненты, оценка параметров MLE |
| Вихрь Мерсенна | [MT19937](#mt) | 1 | собственная реализация, сверка с `numpy` |
| Автотесты датчика | [Random 01](#r1-tests), [Random 02](#r2-tests) | 1 | 13 тестов в `test_generators.py`, запуск `pytest` |
| Табличное представление итоговых метрик | [Random 01](#r1-tests), [Random 02](#r2-code) | 1 | |
| Итеративный подход (версии 1 и 2) | [Random 02](#r2-code) | 0.5 | в ноутбуке видна логика v1 → проблема → v2; история в git не показана |
| За каждое распределение | [Random 01](#r1), [Random 02](#r2) | 1 | дискретное эмпирическое, смесь нормальных, усечённая смесь |
| Список литературы | [Литература](#lit) | 1 | 18 источников |
| Дополнительный датасет | - | 0 | не делали |
| Интерактивность / marimo (для КТвД) | - | 0 | не делали |

<a id="lit"></a>
## Литература
1. Matsumoto M., Nishimura T. Mersenne Twister: a 623-dimensionally equidistributed uniform pseudo-random number generator // ACM Transactions on Modeling and Computer Simulation. 1998. Vol. 8, No. 1. P. 3-30.
2. Knuth D. E. The Art of Computer Programming, Vol. 2: Seminumerical Algorithms. 3rd ed. Addison-Wesley, 1997.
3. Devroye L. Non-Uniform Random Variate Generation. Springer, 1986.
4. Marsaglia G., Bray T. A. A convenient method for generating normal variables // SIAM Review. 1964. Vol. 6, No. 3. P. 260-264.
5. Walker A. J. An efficient method for generating discrete random variables with general distributions // ACM Transactions on Mathematical Software. 1977. Vol. 3, No. 3. P. 253-256.
6. Vose M. D. A linear algorithm for generating random numbers with a given distribution // IEEE Transactions on Software Engineering. 1991. Vol. 17, No. 9. P. 972-975.
7. Box G. E. P., Muller M. E. A note on the generation of random normal deviates // The Annals of Mathematical Statistics. 1958. Vol. 29, No. 2. P. 610-611.
8. Efron B. Bootstrap methods: another look at the jackknife // The Annals of Statistics. 1979. Vol. 7, No. 1. P. 1-26.
9. Mann H. B., Whitney D. R. On a test of whether one of two random variables is stochastically larger than the other // The Annals of Mathematical Statistics. 1947. Vol. 18, No. 1. P. 50-60.
10. D'Agostino R., Pearson E. S. Tests for departure from normality // Biometrika. 1973. Vol. 60, No. 3. P. 613-622.
11. Massey F. J. The Kolmogorov-Smirnov test for goodness of fit // Journal of the American Statistical Association. 1951. Vol. 46, No. 253. P. 68-78.
12. Lilliefors H. W. On the Kolmogorov-Smirnov test for normality with mean and variance unknown // Journal of the American Statistical Association. 1967. Vol. 62, No. 318. P. 399-402.
13. Cochran W. G. Sampling Techniques. 3rd ed. Wiley, 1977.
14. L'Ecuyer P., Simard R. TestU01: a C library for empirical testing of random number generators // ACM Transactions on Mathematical Software. 2007. Vol. 33, No. 4. Art. 22.
15. Гнеденко Б. В. Курс теории вероятностей.
16. Azzalini A. A class of distributions which includes the normal ones // Scandinavian Journal of Statistics. 1985. Vol. 12, No. 2. P. 171-178.
17. Akaike H. A new look at the statistical model identification // IEEE Transactions on Automatic Control. 1974. Vol. 19, No. 6. P. 716-723.
18. von Neumann J. Various techniques used in connection with random digits // Monte Carlo Method (National Bureau of Standards, Applied Mathematics Series 12). 1951. P. 36-38.
